# GraphRAG

## Import packages

In [ ]:
import sys
sys.path.append('..')
sys.path.append('../neurorag')
sys.path.append('../neurorag/chains')

import os
import nltk
import string
import numpy as np
import pandas as pd
from unidecode import unidecode
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
from pathlib import Path
from rouge_score import rouge_scorer
import json
from dotenv import load_dotenv
from getpass import getpass

from langchain_community.embeddings import OllamaEmbeddings
from langchain.embeddings.cache import CacheBackedEmbeddings
from langchain.storage import LocalFileStore

from neurorag.neurorag import NeuroRAG

from metrics import (
  embeddings_cosine_sim_metric,
  bleu_metric,
  rogue_l_metric,
  rogue_1_metric,
  factscore_metric,
)

## Disable warnings

In [2]:
import warnings
warnings.filterwarnings('ignore')

## Setup environment variables

You have to define the following environment variables in the `.env` file, terminal environment, or input field within this Jupyter notebook:
1. MISTRAL_API_KEY
2. OPENAI_API_KEY
3. OPENAI_PROXY
4. TAVILY_API_KEY
5. ENTREZ_EMAIL

## Import packages

In [3]:
env_variables = [
  'MISTRAL_API_KEY',
  'OPENAI_API_KEY',
  'OPENAI_PROXY',
  'TAVILY_API_KEY',
  'ENTREZ_EMAIL',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)

  if value is None:
    value = getpass(key)

  os.environ[key] = value

## Build model

In [11]:
app = NeuroRAG(debug=False)
app.compile()

## Evaluate RAG

### Load QA dataset

In [12]:
mediqa_df = pd.read_csv('../datasets/neurobiology_mediqa.csv')
mediqa_df

,question,answer
0,SSPE. My son is 33years of age and did not hav...,Subacute sclerosing panencephalitis: Subacute ...
1,Homozygout MTHFR A1298C Health Issues and long...,MTHFR gene variant (Inheritance): Because each...
2,What is Stroke?,Stroke: A stroke occurs when the blood supply ...
3,What causes Stroke?,Ischemic Stroke (Summary): Summary A stroke is...
4,What are the symptoms of Stroke?,What are the symptoms of Stroke?: The signs an...
5,What are the treatments of Stroke?,Stroke (Treatment): A stroke is a medical emer...
6,What is Dementia?,Dementia (WHAT IS DEMENTIA?): Dementia is the ...
7,What causes Dementia?,What causes Dementia?: Dementia usually occurs...
8,What are the symptoms of Dementia?,Dementia (Symptoms): Dementia symptoms include...
9,How to diagnose Dementia?,Dementia (Diagnosis): Diagnosing dementia and ...


### Load cached RAGs responses

In [13]:
cache_path = Path('cache.json')

if not os.path.exists(cache_path):
  data = {}
  with open(cache_path, 'w') as file:
    json.dump(data, file)

with open(cache_path, 'r') as f:
  cache = json.load(f)

len(cache.keys())

0

In [ ]:
questions = list(mediqa_df['question'].tolist())
expected_answers = list(mediqa_df['answer'].tolist())
predicted_answers = []

for index, question in tqdm(enumerate(questions)):
  if question not in cache:
    cache[question] = app.invoke({'question': question})['generation']

  predicted_answers.append(cache[question])

  with open(cache_path, 'w') as f:
    json.dump(cache, f)

cos_score = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
bleu_score = bleu_metric(expected_answers, predicted_answers)
rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)

cos_score, bleu_score, rogue_1_score, rogue_l_score

0it [00:00, ?it/s]